# Lab 02 · Retrieve architecture rules with RAG

**Prepared by Mustafa Assaf** · 5–10 minute example

## Retrieve the policy instead of pasting it

**So far:** in Lab 01 we pasted the policy into every prompt.

**New:** keep policies in a small store, *retrieve* the one that matches the question, and give only that text to the model. This is retrieval-augmented generation (RAG).

Our retriever is simple on purpose. It returns a record when the record's topic word appears in the question. Real systems often use embeddings and a vector database, but the idea is the same: find the evidence first, then answer from it.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Store records and retrieve the relevant one

Each record has an ID, a topic, and text. `R1` is our architecture policy. `R2` is an unrelated rule that should *not* be retrieved.

In [ ]:
policies = [
    {"id": "R1", "topic": "architecture",
     "text": "Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise choose modular monolith."},
    {"id": "R2", "topic": "database", "text": "Use transactions when updating account balances."},
]
def retrieve(question):
    words = question.lower().split()
    return [record for record in policies if record["topic"] in words]
question = "Which architecture fits team_size=3 and independent_deploys=False?"
found = retrieve(question)
print("Retrieved:", [record["id"] for record in found])

## Step 2 · Answer from the retrieved text

The retrieved records become the prompt's *context*. If nothing was retrieved, the code does not call the model. It reports that no rule matched. The prompt asks for the rule ID, so the answer can be traced back to its source.

In [ ]:
if found:
    context = "\n".join(f"{record['id']}: {record['text']}" for record in found)
    prompt = f"Use only this policy:\n{context}\n\n{question}\nReply with the choice and the rule ID."
    answer = ask(prompt, example="modular monolith (R1)")
else:
    answer = "No matching rule."  # no evidence, so no model call
print(answer)
assert "R1" in answer, "The answer should cite the retrieved rule R1."

## What you should see

`Retrieved: ['R1']`, then `modular monolith (R1)`. The check confirms that the answer cites R1. Read the answer yourself to confirm that R1 actually supports the choice.

## Try it

Set `question = "Which deployment region should we use?"` and predict what `retrieve` returns. No topic matches, so Step 2 prints `No matching rule.` without calling the model. Change the `assert` to check for that answer.

## Takeaway

Retrieval is only as good as its matching. This keyword retriever misses synonyms, such as "design" for "architecture". When nothing relevant is found, saying so is better than answering without evidence.

## Read further

- [The original RAG paper](https://arxiv.org/abs/2005.11401)